> **Archived precursor.** This exploratory notebook predates the tested trajectory-anomaly lifecycle. It is retained for project history, not as the prescribed or current implementation.

# Week 1 — Data Recon

**Goal:** Answer three questions before we model anything.
1. Is the OpenSky data real and usable for LEMD bounding box?
2. How many tracks survive the alt < 1500m AND velocity < 50 m/s filter?
3. Does anything look plausibly drone-sized (alt < 200m, velocity < 15 m/s)?

**Do not model anything in this notebook.** Just inspect.

**Colab setup:** Runtime → Change runtime type → T4 GPU (not needed here, but good habit).

In [ ]:
# Mount Google Drive (Colab only — skip if running locally)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/drone-ai-saturdays/data'
else:
    DATA_DIR = '../data'

In [ ]:
# Install dependencies (Colab only)
if IN_COLAB:
    !pip install -q impyla thrift thrift_sasl pandas matplotlib seaborn folium

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# LEMD bounding box
LAT_MIN, LAT_MAX = 40.3, 40.6
LON_MIN, LON_MAX = -3.8, -3.5

print('Setup complete')

## Option A — Impala bulk query (requires OpenSky research account)

Register at: https://opensky-network.org/index.php?option=com_users&view=registration

If your account is pending, skip to **Option B** (live REST API).

In [ ]:
# --- Option A: Impala bulk ---
# Uncomment and fill credentials once research account is approved.

# from impala.dbapi import connect
# conn = connect(host='data.opensky-network.org', port=21050,
#                use_ssl=True, auth_mechanism='PLAIN',
#                user='YOUR_USERNAME', password='YOUR_PASSWORD')
# cursor = conn.cursor()
#
# SQL = """
# SELECT icao24, time, lat, lon, baroaltitude, geoaltitude, velocity, heading, vertrate, onground
# FROM state_vectors_data4
# WHERE lat BETWEEN {lat_min} AND {lat_max}
#   AND lon BETWEEN {lon_min} AND {lon_max}
#   AND hour >= 1704067200  -- 2024-01-01
#   AND hour <  1706745600  -- 2024-02-01
#   AND time % 10 = 0       -- one state vector per 10s (reduce volume)
# """.format(lat_min=LAT_MIN, lat_max=LAT_MAX, lon_min=LON_MIN, lon_max=LON_MAX)
#
# cursor.execute(SQL)
# df_raw = pd.DataFrame(cursor.fetchall(), columns=[d[0] for d in cursor.description])
# df_raw.to_parquet(f'{DATA_DIR}/raw/lemd_jan2024.parquet', index=False)
# print(f'Saved {len(df_raw):,} rows')

## Option B — Live REST API (no account needed, use while waiting for Impala)

Only covers current live traffic — enough for EDA shape, not for training data volume.

In [ ]:
import requests, time

def fetch_lemd_states():
    url = 'https://opensky-network.org/api/states/all'
    params = dict(lamin=LAT_MIN, lamax=LAT_MAX, lomin=LON_MIN, lomax=LON_MAX)
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    data = r.json()
    if not data or not data.get('states'):
        return pd.DataFrame()
    cols = ['icao24','callsign','origin_country','time_position','last_contact',
            'lon','lat','baro_altitude','on_ground','velocity','true_track',
            'vertical_rate','sensors','geo_altitude','squawk','spi','position_source']
    return pd.DataFrame(data['states'], columns=cols)

# Collect 5 snapshots × 10s apart for a small live sample
snapshots = []
for i in range(5):
    snap = fetch_lemd_states()
    if not snap.empty:
        snap['snapshot'] = i
        snapshots.append(snap)
    print(f'Snapshot {i+1}: {len(snap)} tracks')
    if i < 4:
        time.sleep(10)

df_live = pd.concat(snapshots, ignore_index=True) if snapshots else pd.DataFrame()
print(f'\nTotal rows: {len(df_live)}')
df_live.head()

## EDA — answer the three recon questions

In [ ]:
# Load data — use Impala parquet if available, otherwise use live snapshot
parquet_path = f'{DATA_DIR}/raw/lemd_jan2024.parquet'
if os.path.exists(parquet_path):
    df = pd.read_parquet(parquet_path)
    print(f'Loaded bulk data: {len(df):,} rows, {df["icao24"].nunique():,} unique tracks')
elif not df_live.empty:
    df = df_live.rename(columns={'baro_altitude': 'baroaltitude',
                                  'geo_altitude': 'geoaltitude',
                                  'true_track': 'heading',
                                  'vertical_rate': 'vertrate',
                                  'on_ground': 'onground'})
    print(f'Using live snapshot: {len(df):,} rows')
else:
    raise RuntimeError('No data available — run Option A or Option B first')

In [ ]:
# Q1: How many unique tracks?
n_tracks = df['icao24'].nunique()
print(f'Unique ICAO24 tracks: {n_tracks:,}')
print(f'Total state vectors: {len(df):,}')
print(f'Avg vectors per track: {len(df)/n_tracks:.1f}')

In [ ]:
# Q2: Altitude and speed distributions
alt_col = 'baroaltitude' if 'baroaltitude' in df.columns else 'baro_altitude'
vel_col = 'velocity'

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df[alt_col].dropna().clip(0, 5000).hist(bins=50, ax=axes[0], edgecolor='none')
axes[0].axvline(1500, color='red', linestyle='--', label='1500m filter')
axes[0].set_xlabel('Baro altitude (m)')
axes[0].set_title('Altitude distribution')
axes[0].legend()

df[vel_col].dropna().clip(0, 200).hist(bins=50, ax=axes[1], edgecolor='none')
axes[1].axvline(50, color='red', linestyle='--', label='50 m/s filter')
axes[1].set_xlabel('Velocity (m/s)')
axes[1].set_title('Speed distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{DATA_DIR}/../docs/weekly/figures/week1_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Q2: How many tracks survive the filter?
mask = (
    (df[alt_col].notna()) & (df[alt_col] < 1500) &
    (df[vel_col].notna()) & (df[vel_col] < 50)
)
df_filtered = df[mask]
filtered_tracks = df_filtered['icao24'].nunique()
print(f'After alt<1500m AND vel<50m/s filter:')
print(f'  Rows: {len(df_filtered):,} ({100*len(df_filtered)/len(df):.1f}% of total)')
print(f'  Unique tracks: {filtered_tracks:,}')

In [ ]:
# Q3: Drone-sized candidates (alt < 200m AND velocity < 15 m/s)
drone_mask = (
    (df[alt_col].notna()) & (df[alt_col] < 200) &
    (df[vel_col].notna()) & (df[vel_col] < 15)
)
df_drone_candidates = df[drone_mask]
drone_tracks = df_drone_candidates['icao24'].nunique()
print(f'Drone-sized candidates (alt<200m AND vel<15m/s):')
print(f'  Rows: {len(df_drone_candidates):,}')
print(f'  Unique ICAO24s: {drone_tracks:,}')
if drone_tracks > 0:
    print('\nSample:')
    print(df_drone_candidates[['icao24', alt_col, vel_col]].drop_duplicates('icao24').head(10))

In [ ]:
# Track length distribution (state vectors per ICAO24)
track_lengths = df.groupby('icao24').size()
print(track_lengths.describe())

track_lengths.clip(0, 500).hist(bins=50)
plt.axvline(10, color='red', linestyle='--', label='Min segment length (10 steps)')
plt.xlabel('State vectors per track')
plt.title('Track length distribution')
plt.legend()
plt.show()

usable_tracks = (track_lengths >= 10).sum()
print(f'Tracks with >= 10 state vectors: {usable_tracks:,}')

## Summary — paste this into the team Discord

Run the cell below to print a share-ready summary.

In [ ]:
print('=== LEMD Data Recon Summary ===')
print(f'Data source: {"Impala bulk" if os.path.exists(parquet_path) else "Live API (5 snapshots)"}')
print(f'Total rows: {len(df):,}')
print(f'Unique tracks: {df["icao24"].nunique():,}')
print(f'After alt<1500m AND vel<50m/s: {filtered_tracks:,} tracks')
print(f'Drone-sized candidates (alt<200m, vel<15m/s): {drone_tracks:,} tracks')
print(f'Tracks with >=10 state vectors: {usable_tracks:,}')
print()
if filtered_tracks >= 500:
    print('STATUS: GOOD — enough data to proceed to Week 2 pipeline')
elif filtered_tracks >= 100:
    print('STATUS: MARGINAL — consider widening bbox or extending time range (see design doc Open Question 2)')
else:
    print('STATUS: INSUFFICIENT — trigger fallback in design doc Open Question 2')